# Nextgen AI - Seq2Seq Uretici (Colab / PyTorch + GPU)

Amac: **(kullanici sorgusu) -> (dogal Turkce yanit)** iliskisini ogrenen karakter-seviyesi transformer **encoder-decoder** uretecin GPU ile egitimi.

- Mimari `seq2seq.py` (NumPy inference) ile **birebir ayni**: paylasilan char embedding, sinusoidal konum, N encoder blogu + N decoder blogu (causal self-attn + cross-attn + GELU FFN), final LN + lineer kafa.
- Egitim verisi `intents.json` -> (pattern, response) ciftleri (`seqgen.load_pairs`, sorgu-koullu).
- Cikti: `model/seq2seq_model.json` -> yerelde `seq2seq.load_seq2seq()`; `brain._try_seq_rephrase` once bunu dener, yoksa LSTM'e duser.

## Calistirma sirasi
1. `/content` altina suklayin: `intents.json`, `seqgen.py`, `seq2seq.py`
2. Tum hucreleri calistirin (Ctrl+F9).
3. Son hucredeki indirme talimatini izleyin; `model/seq2seq_model.json`'i yerel `model/` klasorune kopyalayin.


In [ ]:
import sys, os, io, json, math, time, random
import numpy as np
import torch
import torch.nn as nn
sys.path.insert(0, '/content')

from google.colab import drive
USE_DRIVE = False                        # True yaparsan Drive mount + checkpoint istenir
try:
    if USE_DRIVE:
        drive.mount('/content/drive')
        DRIVE_DIR = '/content/drive/MyDrive/NextgenAI'
    else:
        DRIVE_DIR = os.path.join(os.getcwd(), 'ckpt')
except Exception as e:
    print('Drive mount yapilamadi, yerel checkpoint kullanilacak:', e)
    DRIVE_DIR = os.path.join(os.getcwd(), 'ckpt')
os.makedirs(DRIVE_DIR, exist_ok=True)

import seqgen
from seqgen import clean_chars, build_vocab, load_pairs
import seq2seq
from seq2seq import Seq2Seq, encode_seq2, load_seq2seq, save_seq2seq

# ---------------- hiperparametreler (numpy inference ile AYNI mimari)
TARGET_MAX_LEN = 42    # hedef yanitlar tam cumle + bu uzunlukla sinirli
D_MODEL     = 128       # embedding boyutu
NUM_BLOCKS  = 4        # encoder + decoder blok sayisi
NUM_HEADS   = 4
FF_MULT     = 3        # FFN genisletme
DROPOUT     = 0.10
BATCH_SIZE  = 64
EPOCHS      = 400
LR_BASE     = 1e-3
LR_MIN      = 0.1      # cosine alt siniri (LR_BASE orani)
WARMUP      = 200      # adim
PATIENCE    = 20       # erken durdurma
GRAD_CLIP   = 5.0
CKPT_FREQ   = 5        # kac epoch'ta bir checkpoint
MAX_PAIRS   = 20000
MAX_ENC_LEN = 40
MAX_DEC_LEN = 48
SEED        = 7

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '-')


In [ ]:
# intents.json -> (sorgu, yanit) ciftleri; 10% val ayrimi.
INTENTS = '/content/intents.json'
assert os.path.exists(INTENTS), 'intents.json yuklenmedi!'

pairs = load_pairs(INTENTS, max_pairs=MAX_PAIRS, use_query=True)
print('egitim cifti (sorgu, yanit):', len(pairs))

def refine_resp(r, maxc=TARGET_MAX_LEN):
    r = (r or '').strip()
    if len(r) < 10:
        return None
    if len(r) > maxc:
        cut = 0
        for i in range(15, min(len(r), maxc + 8)):
            if r[i] in '.?:!':
                cut = i
        r = r[:cut + 1].strip() if cut > 0 else r[:maxc].strip()
    if r and r[-1] not in '.?!...':
        r = r + '.'
    return r if len(r) >= 10 else None

pairs = [(ctx, rr) for ctx, r in pairs if (rr := refine_resp(r)) is not None]
print('rafine hedef sonrasi cift:', len(pairs))

all_text = []
for ctx, resp in pairs:
    all_text.append(ctx); all_text.append(resp)
vocab = build_vocab(all_text)
print('karakter sozlugu:', len(vocab))

dummy = Seq2Seq(vocab, d_model=4, num_blocks=1, num_heads=1,   # encode icin min model
                max_enc_len=MAX_ENC_LEN, max_dec_len=MAX_DEC_LEN, seed=SEED)

rng = np.random.RandomState(SEED)
perm = rng.permutation(len(pairs))
n_val = max(1, int(0.1 * len(pairs)))
tr_pairs = [pairs[i] for i in perm[n_val:]]
va_pairs = [pairs[i] for i in perm[:n_val]]

def make_batches(pairs_, B, dec_len=MAX_DEC_LEN):
    items = sorted(pairs_, key=lambda pr: len(pr[0]))
    batches = []
    for i in range(0, len(items), B):
        block = items[i:i + B]
        encs, dins, dtgts, dmasks = [], [], [], []
        maxe = 0
        for ctx, resp in block:
            e, d1, d2, m1, m2 = encode_seq2(dummy, ctx, resp, max_dec=dec_len)
            encs.append(e); dins.append(d1); dtgts.append(d2); dmasks.append(m2)
            maxe = max(maxe, len(e))
        E = np.zeros((len(block), maxe), np.int64)
        EM = np.zeros((len(block), maxe), np.float32)
        for j, e in enumerate(encs):
            E[j, :len(e)] = e; EM[j, :len(e)] = 1.0
        DI = np.stack(dins); DT = np.stack(dtgts); DM = np.stack(dmasks)
        batches.append((E, DI, DT, EM, DM))
    return batches

tr = make_batches(tr_pairs, BATCH_SIZE)
va = make_batches(va_pairs, BATCH_SIZE)
print('train batch:', len(tr), '| val batch:', len(va))
print('ornek cift:', tr_pairs[0])


In [ ]:
class TorchSeq2Seq(nn.Module):
    """seq2seq.Seq2Seq ile birebir AYNI cebir (PyTorch). Parametre adlari
    numpy 'params' anahtarlariyla eslesecek sekilde duz attribute olarak tutulur
    (b0_Wq, db0_Wqc, head, embed ...). Dropout yalnizca training'de; eval'da
    kapali -> NumPy parity bozulmaz."""
    def __init__(self, V, d_model=96, num_blocks=3, num_heads=3, ff_mult=3,
                 max_enc_len=40, max_dec_len=80, drop=0.10):
        super().__init__()
        self.V, self.D = V, d_model
        self.N, self.H = num_blocks, num_heads
        self.hd = d_model // num_heads
        self.ff = ff_mult * d_model
        self.rsqrt = self.hd ** -0.5
        self.drop = drop

        def he(shape, scale=None):
            if scale is None:
                scale = math.sqrt(2.0 / shape[0])
            return torch.randn(*shape) * scale

        with torch.no_grad():
            self.embed = nn.Parameter(torch.randn(V, d_model) * 0.02)
            self.embed.data[0].zero_()            # PAD satiri sabit 0
            pe = torch.zeros(max(max_enc_len, max_dec_len), d_model)
            pos = torch.arange(max(max_enc_len, max_dec_len), dtype=torch.float32).unsqueeze(1)
            dim = torch.arange(d_model // 2, dtype=torch.float32)
            div = 10000.0 ** (2.0 * dim / d_model)
            pe[:, 0::2] = torch.sin(pos / div)
            pe[:, 1::2] = torch.cos(pos / div)
            pe = pe / math.sqrt(max(d_model, 1))
        self.register_buffer('pos_enc', pe)

        for i in range(num_blocks):
            for n, dims in [('Wq', (d_model, d_model)), ('Wk', (d_model, d_model)),
                            ('Wv', (d_model, d_model)), ('Wo', (d_model, d_model))]:
                setattr(self, f'b{i}_{n}', nn.Parameter(he(dims)))
                setattr(self, f'b{i}_b{n[1:]}', nn.Parameter(torch.zeros(1, d_model)))
            setattr(self, f'b{i}_ln1_g', nn.Parameter(torch.ones(1, d_model)))
            setattr(self, f'b{i}_ln1_b', nn.Parameter(torch.zeros(1, d_model)))
            setattr(self, f'b{i}_W1', nn.Parameter(he((d_model, self.ff))))
            setattr(self, f'b{i}_b1', nn.Parameter(torch.zeros(1, self.ff)))
            setattr(self, f'b{i}_W2', nn.Parameter(he((self.ff, d_model), scale=0.02)))
            setattr(self, f'b{i}_b2', nn.Parameter(torch.zeros(1, d_model)))
            setattr(self, f'b{i}_ln2_g', nn.Parameter(torch.ones(1, d_model)))
            setattr(self, f'b{i}_ln2_b', nn.Parameter(torch.zeros(1, d_model)))
            for n, dims in [('Wq', (d_model, d_model)), ('Wk', (d_model, d_model)),
                            ('Wv', (d_model, d_model)), ('Wo', (d_model, d_model))]:
                setattr(self, f'db{i}_{n}', nn.Parameter(he(dims)))
                setattr(self, f'db{i}_b{n[1:]}', nn.Parameter(torch.zeros(1, d_model)))
            setattr(self, f'db{i}_ln1_g', nn.Parameter(torch.ones(1, d_model)))
            setattr(self, f'db{i}_ln1_b', nn.Parameter(torch.zeros(1, d_model)))
            for n, dims in [('Wq', (d_model, d_model)), ('Wk', (d_model, d_model)),
                            ('Wv', (d_model, d_model)), ('Wo', (d_model, d_model))]:
                setattr(self, f'db{i}_{n}c', nn.Parameter(he(dims)))
                setattr(self, f'db{i}_b{n[1:]}c', nn.Parameter(torch.zeros(1, d_model)))
            setattr(self, f'db{i}_ln2_g', nn.Parameter(torch.ones(1, d_model)))
            setattr(self, f'db{i}_ln2_b', nn.Parameter(torch.zeros(1, d_model)))
            setattr(self, f'db{i}_W1', nn.Parameter(he((d_model, self.ff))))
            setattr(self, f'db{i}_b1', nn.Parameter(torch.zeros(1, self.ff)))
            setattr(self, f'db{i}_W2', nn.Parameter(he((self.ff, d_model), scale=0.02)))
            setattr(self, f'db{i}_b2', nn.Parameter(torch.zeros(1, d_model)))
            setattr(self, f'db{i}_ln3_g', nn.Parameter(torch.ones(1, d_model)))
            setattr(self, f'db{i}_ln3_b', nn.Parameter(torch.zeros(1, d_model)))

        self.out_ln_g = nn.Parameter(torch.ones(1, d_model))
        self.out_ln_b = nn.Parameter(torch.zeros(1, d_model))
        self.head = nn.Parameter(he((d_model, V), scale=0.02))
        self.head_b = nn.Parameter(torch.zeros(1, V))

    def _ln(self, x, g, b):
        return torch.nn.functional.layer_norm(
            x, (x.size(-1),), g.reshape(-1), b.reshape(-1))

    @staticmethod
    def _attn(q, k, v, rsqrt, heads, hd, mask=None, causal=False,
              drop_p=0.0, training=False):
        B, T, d = q.shape
        S = k.shape[1]
        def split(x):
            return x.reshape(B, x.shape[1], heads, hd).transpose(1, 2)
        Q, K, V = split(q), split(k), split(v)
        scores = (Q @ K.transpose(-1, -2)) * rsqrt
        if mask is not None:
            scores = scores.masked_fill((mask <= 0)[:, None, None, :], -1e9)
        if causal:
            tri = torch.triu(torch.full((T, T), -1e9, device=q.device), 1)
            scores = scores + tri[None, None]
        p = torch.softmax(scores, dim=-1)
        if training and drop_p > 0:
            p = torch.nn.functional.dropout(p, drop_p)
        return (p @ V).transpose(1, 2).reshape(B, T, d)

    def _ffn(self, x, W1, b1, W2, b2):
        return torch.nn.functional.gelu(x @ W1 + b1, approximate='tanh') @ W2 + b2

    def forward(self, enc, dec):
        B, T1 = enc.shape
        T2 = dec.shape[1]
        em = (enc != 0).float()
        x = self.embed[enc] * em.unsqueeze(-1)
        x = x + self.pos_enc[:T1][None] * em.unsqueeze(-1)
        if self.training and self.drop > 0:
            x = torch.nn.functional.dropout(x, self.drop)
        for i in range(self.N):
            pre = self._ln(x, getattr(self, f'b{i}_ln1_g'), getattr(self, f'b{i}_ln1_b'))
            a = self._attn(pre, pre, pre, self.rsqrt, self.H, self.hd, mask=em,
                          drop_p=0.05, training=self.training)
            x = x + a
            pre = self._ln(x, getattr(self, f'b{i}_ln2_g'), getattr(self, f'b{i}_ln2_b'))
            x = x + self._ffn(pre, getattr(self, f'b{i}_W1'), getattr(self, f'b{i}_b1'),
                             getattr(self, f'b{i}_W2'), getattr(self, f'b{i}_b2'))
        enc_out = x

        y = self.embed[dec]
        y = y + self.pos_enc[:T2][None]
        if self.training and self.drop > 0:
            y = torch.nn.functional.dropout(y, self.drop)
        for i in range(self.N):
            pre = self._ln(y, getattr(self, f'db{i}_ln1_g'), getattr(self, f'db{i}_ln1_b'))
            a = self._attn(pre, pre, pre, self.rsqrt, self.H, self.hd,
                          causal=True, drop_p=0.05, training=self.training)
            y = y + a
            pre = self._ln(y, getattr(self, f'db{i}_ln2_g'), getattr(self, f'db{i}_ln2_b'))
            c = self._attn(pre, enc_out, enc_out, self.rsqrt, self.H, self.hd,
                          mask=em, drop_p=0.05, training=self.training)
            y = y + c
            pre = self._ln(y, getattr(self, f'db{i}_ln3_g'), getattr(self, f'db{i}_ln3_b'))
            y = y + self._ffn(pre, getattr(self, f'db{i}_W1'), getattr(self, f'db{i}_b1'),
                             getattr(self, f'db{i}_W2'), getattr(self, f'db{i}_b2'))
        h = self._ln(y, self.out_ln_g, self.out_ln_b)
        return h @ self.head + self.head_b

def seq2_loss(logits, tgt, mask):
    lg = torch.log_softmax(logits, dim=-1)
    nll = lg.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    return -(nll * mask).sum() / mask.sum().clamp(min=1.0)

def masked_acc(logits, tgt, mask):
    ok = ((logits.argmax(-1) == tgt) & mask.bool())
    return ok.sum().item() / mask.sum().clamp(min=1.0).item()


In [ ]:
trE = [torch.from_numpy(b[0]).long().to(DEVICE) for b in tr]
trI = [torch.from_numpy(b[1]).long().to(DEVICE) for b in tr]
trT = [torch.from_numpy(b[2]).long().to(DEVICE) for b in tr]
trM = [torch.from_numpy(b[4]).float().to(DEVICE) for b in tr]
vaE = [torch.from_numpy(b[0]).long().to(DEVICE) for b in va]
vaI = [torch.from_numpy(b[1]).long().to(DEVICE) for b in va]
vaT = [torch.from_numpy(b[2]).long().to(DEVICE) for b in va]
vaM = [torch.from_numpy(b[4]).float().to(DEVICE) for b in va]

CKPT = os.path.join(DRIVE_DIR, 'seq2seq_ckpt.pt')

model = TorchSeq2Seq(len(vocab), D_MODEL, NUM_BLOCKS, NUM_HEADS, FF_MULT,
                     MAX_ENC_LEN, MAX_DEC_LEN, drop=DROPOUT).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR_BASE)

best_state = None
best_val = 1e9
bad = 0
start_ep = 0
step = 0
tot_steps = EPOCHS * len(trE)

if os.path.exists(CKPT):                      # kesinti sonrasi kaldigi yerden
    cp = torch.load(CKPT, map_location=DEVICE, weights_only=True)
    model.load_state_dict(cp['model']); opt.load_state_dict(cp['opt'])
    best_val, best_state, start_ep, step = cp['best_val'], cp['best_state'], cp['epoch'], cp['step']
    best_state = {k: v.detach().cpu().clone() for k, v in best_state.items()}
    print('Devam: epoch', start_ep, '| step', step, '| best val:', round(best_val, 4))


In [ ]:
t0_all = time.time()
done = False
for ep in range(start_ep + 1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    order = list(range(len(trE))); random.Random(ep).shuffle(order)
    tl = 0.0
    for bi in order:
        step += 1
        cur = LR_BASE
        if step <= WARMUP:
            cur = LR_BASE * (step / WARMUP)
        else:
            prog = (step - WARMUP) / max(1, tot_steps - WARMUP)
            cur = LR_BASE * (LR_MIN + (1 - LR_MIN) * 0.5 * (1 + math.cos(math.pi * prog)))
        for g in opt.param_groups:
            g['lr'] = cur
        opt.zero_grad()
        loss = seq2_loss(model(trE[bi], trI[bi]), trT[bi], trM[bi])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
        tl += loss.item()
    tl /= len(trE)

    model.eval(); vl = va_acc = 0.0
    with torch.no_grad():
        for e, i, t, m in zip(vaE, vaI, vaT, vaM):
            lg = model(e, i)
            vl += seq2_loss(lg, t, m).item()
            va_acc += masked_acc(lg, t, m)
    vl /= len(vaE); va_acc /= len(vaE)
    print(f'epoch {ep:3d}/{EPOCHS} | train {tl:.4f} | val {vl:.4f} | acc {va_acc:.3f} | {time.time()-t0:.1f}s | lr {cur:.5f}', flush=True)

    if vl < best_val - 1e-4:
        best_val = vl
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f'[seq2seq] Erken durdurma. Best val: {best_val:.4f}')
            done = True
    if ep % CKPT_FREQ == 0 or done:
        torch.save({'epoch': ep, 'step': step, 'model': best_state,
                    'opt': opt.state_dict(), 'best_val': best_val,
                    'best_state': best_state}, CKPT)
        print(f'  checkpoint -> {CKPT}')
    if done:
        break

print('\nToplam egitim suresi: %.1f dk' % ((time.time() - t0_all) / 60))


In [ ]:
model.load_state_dict(best_state); model.eval()
best_state = {k: v.detach().cpu().clone() for k, v in best_state.items()}

out_dir = '/content/model'
os.makedirs(out_dir, exist_ok=True)

def export(state):
    data = {
        'arch': 'seq2seq',
        'V': len(VOCAB),
        'd_model': D_MODEL, 'num_blocks': NUM_BLOCKS, 'num_heads': NUM_HEADS,
        'ff_mult': FF_MULT,
        'max_enc_len': MAX_ENC_LEN, 'max_dec_len': MAX_DEC_LEN,
        'vocab': VOCAB,
        'params': {k: v.numpy().tolist() for k, v in state.items()},
    }
    return data

VOCAB = vocab  # build_vocab ciktisi (PAD/BOS/EOS onday)
data = export(best_state)
dest = os.path.join(out_dir, 'seq2seq_model.json')
with open(dest, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False)
with open(os.path.join(DRIVE_DIR, 'seq2seq_model.json'), 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False)   # Drive yedegi
print('model/seq2seq_model.json yazildi:', dest)


In [ ]:
# PARITY: PyTorch egitimli model ile numpy Seq2Seq birebir ayni mi?
m = Seq2Seq(['<PAD>', '<BOS>', '<EOS>'])
with open(dest, encoding='utf-8') as f:
    dj = json.load(f)
m.from_dict(dj)

maxdiff = 0.0
for k in m.params:
    t = best_state[k].float().numpy()
    n = m.params[k]
    if t.shape != n.shape:
        raise AssertionError(f'shape uyumsuz: {k} torch={t.shape} numpy={n.shape}')
    maxdiff = max(maxdiff, float(np.abs(t - n).max()))
print('param maxdiff:', maxdiff)
assert maxdiff < 1e-6

# Ayni girdiyle ileri gecis: torch eval vs numpy
ctx, resp = tr_pairs[0]
e, d1, d2, m1, m2 = encode_seq2(m, ctx, resp, max_dec=MAX_DEC_LEN)
with torch.no_grad():
    tl = model(torch.from_numpy(e[None]).long().to(DEVICE),
               torch.from_numpy(d1[None]).long().to(DEVICE))[0].float().cpu().numpy()
eo, em = m._encode(e[None])
nl = m._decoder_logits(eo, em, d1[None])[0]
print('forward maxdiff:', float(np.abs(tl - nl).max()))
assert float(np.abs(tl - nl).max()) < 1e-3, 'PARITY FAIL'
print('PARITY OK')


In [ ]:
print('=== SEQ2SEQ ORNEKLERI (temperature 0.6, top_k 6) ===')
for q in ['hava nasil olacak', 'yapay zeka nedir', 'tesekkur ederim',
          'en iyi film hangisi', 'kedi bakimi nasil olur', 'okul ne zaman kapanir']:
    print(f'? {q}\nAI: {m.sample(q, temperature=0.6, top_k=6)}\n')


## Indirme & yerel kurulum

1. `model/seq2seq_model.json`'i indir: `files.download('/content/model/seq2seq_model.json')`
2. Dosyayi `Nextgen_API/model/seq2seq_model.json` olarak kopyala.
3. Yerelde `brain.py` artik once seq2seq'i (transformet encoder-decoder), yoksa LSTM'i dener; kalite kapisindan gecemeyen cikti canned cevaba duser.
4. Manuel test:

```python
from seq2seq import load_seq2seq
m = load_seq2seq()
print(m.sample('hava nasil olacak', temperature=0.6, top_k=6))
```
